<a href="https://colab.research.google.com/github/arctecnologia/AIFraudDetection/blob/main/AIFraudDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Resumo Executivo: Valor de Negócio do Pipeline de Detecção de Anomalias**


# A implementação deste pipeline preditivo para detecção de fraudes em cartões de crédito transcende a simples automação tecnológica, entregando vantagens competitivas e financeiras diretas para a instituição. Ao utilizar algoritmos avançados de aprendizado de máquina adaptados para cenários de alta assimetria de dados, o projeto entrega valor em quatro pilares fundamentais:



# 1. Redução Direta de Perdas Financeiras
Mitigação de Fraudes em Tempo Hábil: O modelo identifica padrões não lineares e complexos que regras de negócio tradicionais (sistemas baseados em if/else) costumam deixar passar. Ao isolar as anomalias com rapidez, a instituição bloqueia transações ilícitas antes que o prejuízo seja consolidado.
Proteção do Limite de Crédito: Reduz a exposição da carteira de crédito da empresa a ataques coordenados e fraudes sequenciais.

# 2. Preservação da Experiência do Cliente (Foco em Precision)
Redução de Falsos Positivos: Bloquear o cartão de um cliente durante uma transação legítima (como um jantar ou uma viagem) gera alto atrito e risco de cancelamento (churn). A arquitetura do modelo foi otimizada para equilibrar a detecção de fraudes com uma alta precisão, garantindo que os alertas gerados sejam de fato assertivos e minimizando o bloqueio indevido de bons clientes.

# 3. Eficiência Operacional e Ganho de Escala
Otimização da Fila de Análise: Em vez de depender de revisões manuais em grandes volumes de transações suspeitas, o modelo atua como um filtro primário inteligente. Isso permite que a equipe de prevenção a fraudes foque apenas nos casos mais críticos e complexos, reduzindo custos operacionais (FTE).
Escalabilidade Tecnológica: Desenvolvido em Python e estruturado para fácil produtização (via APIs), o algoritmo processa grandes volumes de dados com baixo custo computacional.

# 4. Segurança, Privacidade e Compliance
Governança de Dados Sensíveis: O pipeline foi construído desde o início para operar com dados anonimizados via transformação PCA (Componentes Principais). Isso garante que o modelo mantenha alta performance sem expor Informações Pessoalmente Identificáveis (PII), assegurando total conformidade com a LGPD e outras regulamentações bancárias.



# **Configuração do Ambiente e Bibliotecas**

Nesta etapa, importamos as bibliotecas essenciais para manipulação de dados, visualização e modelagem.

In [1]:
# Célula 1 no Colab
# Lembrete: Para download de dados do Kaggle, certifique-se de ter feito o upload do arquivo kaggle.json para a raiz do Colab.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Bibliotecas de Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, auc

# Configurações visuais padrão
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("Bibliotecas carregadas com sucesso. Ambiente pronto.")

Bibliotecas carregadas com sucesso. Ambiente pronto.


# **Ingestão de Dados (Data Sourcing)**

Para utilizar os dados do Kaggle diretamente no Colab, a melhor prática é usar a API oficial.

Nota Operacional: Antes de rodar este bloco, você precisará ter o arquivo kaggle.json (gerado na sua conta do Kaggle em Settings > Create New API Token) e fazer o upload dele para a raiz do Colab

In [ ]:
# Célula 2 no Colab
!pip install -q kaggle

# Cria o diretório para a API do Kaggle e move o token
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download do dataset
!kaggle datasets download -d mlg-ulb/creditcardfraud
!unzip -o creditcardfraud.zip

# Carregamento no Pandas
df = pd.read_csv('creditcard.csv')
print(f"Dataset carregado! Dimensões: {df.shape[0]} linhas e {df.shape[1]} colunas.")
df.head()

# **Profiling e Qualidade de Dados (EDA)**
Antes da modelagem, precisamos entender a distribuição e a qualidade do ativo de dados. Em arquiteturas corporativas orientadas a dados, essa etapa de Data Profiling é crucial para garantir a confiabilidade do produto final.

In [ ]:
# Célula 3 no Colab
# Verificando valores nulos (Completude)
print("Valores ausentes por coluna:\n", df.isnull().sum().max())

# Analisando o desbalanceamento das classes (0 = Normal, 1 = Fraude/Anomalia)
class_counts = df['Class'].value_counts()
fraud_rate = (class_counts[1] / df.shape[0]) * 100

print(f"\nTransações Normais: {class_counts[0]}")
print(f"Anomalias (Fraudes): {class_counts[1]}")
print(f"Taxa de Fraude: {fraud_rate:.3f}%")

# Visualização da distribuição
sns.countplot(x='Class', data=df, palette='Set2')
plt.title('Distribuição de Classes (0: Normal, 1: Anomalia)')
plt.yscale('log') # Escala logarítmica devido ao desbalanceamento
plt.ylabel('Contagem (Log)')
plt.show()

# **Pré-processamento e Transformação**

As variáveis V1 a V28 já estão escalonadas via PCA (uma transformação comum por motivos de privacidade bancária). No entanto, as colunas Time e Amount (Valor) estão em suas escalas originais e precisam ser normalizadas para que o algoritmo não dê peso indevido a grandes valores monetários.

In [ ]:
# Célula 4 no Colab
# Padronizando as colunas 'Amount' e 'Time'
scaler = StandardScaler()
df['Scaled_Amount'] = scaler.fit_transform(df['Amount'].values.reshape(-1,1))
df['Scaled_Time'] = scaler.fit_transform(df['Time'].values.reshape(-1,1))

# Removendo as colunas originais e reordenando
df.drop(['Time', 'Amount'], axis=1, inplace=True)
df.insert(0, 'Scaled_Amount', df.pop('Scaled_Amount'))
df.insert(1, 'Scaled_Time', df.pop('Scaled_Time'))

# Divisão em Treino e Teste
X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Dados particionados: Treino contém", X_train.shape[0], "amostras.")

# **Modelagem Preditiva (Isolation Forest)**

Para detecção de anomalias, algoritmos não supervisionados ou semi-supervisionados geralmente performam melhor do que classificadores tradicionais que sofrem com o desbalanceamento. O Isolation Forest é excelente para isso, pois ele literalmente "isola" anomalias construindo árvores de decisão aleatórias (anomalias requerem menos quebras para serem isoladas).

In [9]:
# Célula 5 no Colab
# Instanciando o modelo
# contamination define a proporção esperada de anomalias
model_if = IsolationForest(n_estimators=100,
                           max_samples=len(X_train),
                           contamination=fraud_rate/100,
                           random_state=42,
                           n_jobs=-1) # Usa todos os núcleos de processamento

print("Treinando o modelo Isolation Forest. Isso pode levar alguns instantes...")
model_if.fit(X_train)

# Predições no conjunto de teste
# O Isolation Forest retorna -1 para anomalias e 1 para dados normais
y_pred_test = model_if.predict(X_test)

# Convertendo as predições para o padrão do dataset (0 para normal, 1 para anomalia)
y_pred_test = np.where(y_pred_test == 1, 0, 1)
print("Treinamento e predições concluídos.")

Treinando o modelo Isolation Forest. Isso pode levar alguns instantes...
Treinamento e predições concluídos.


# **Avaliação de Métricas do Produto**

A Acurácia é uma métrica ilusória em cenários de desbalanceamento extremo (chutar que tudo é "normal" daria 99,8% de acurácia, mas o modelo seria inútil). Precisamos focar no Recall (quantas fraudes reais capturamos) e na Precision (quantos dos nossos alertas são de fato fraudes).

In [ ]:
# Célula 6 no Colab (Relatório de Classificação e Matriz de Confusão)
print("--- Relatório de Classificação ---")
print(classification_report(y_test, y_pred_test, target_names=['Normal', 'Anomalia']))

# Matriz de Confusão
cm = confusion_matrix(y_test, y_pred_test)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Previsto: Normal', 'Previsto: Anomalia'],
            yticklabels=['Real: Normal', 'Real: Anomalia'])
plt.title('Matriz de Confusão - Isolation Forest')
plt.tight_layout()
plt.show()

In [ ]:
# Célula 7 no Colab (Curva Precision-Recall)
# Curva Precision-Recall (ideal para dados desbalanceados)
# Usando a função de decisão do modelo para gerar scores contínuos
scores_test = model_if.decision_function(X_test)
# Invertendo o sinal, pois o scikit-learn pontua anomalias mais baixo
precision, recall, _ = precision_recall_curve(y_test, -scores_test)
pr_auc = auc(recall, precision)

plt.figure(figsize=(10,7))
plt.plot(recall, precision, marker='.', label=f'Isolation Forest (PR AUC = {pr_auc:.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Curva Precision-Recall')
plt.legend()
plt.tight_layout()
plt.show()

# Conclusão Estratégica:
Este produto de dados transforma o setor de prevenção a fraudes de um centro de custos reativo para um escudo de proteção proativo. Ele protege a receita da empresa, otimiza o tempo dos analistas e garante uma jornada de pagamento fluida e invisível para o cliente final.